In [16]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = "/content/drive/MyDrive"
DATA_PATH = os.path.join(PROJECT_ROOT, "Motor_Vehicle_Collisions_-_Crashes_20250104.csv")
MODEL_PATH = os.path.join(PROJECT_ROOT, "nyc_traffic_project/models/crash_predictor.pkl")
OUTPUT_PATH = os.path.join(PROJECT_ROOT, "nyc_traffic_project/outputs")

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

print("✅ Drive mounted!")
print(f"📁 Data path: {DATA_PATH}")
print(f"📁 Model path: {MODEL_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted!
📁 Data path: /content/drive/MyDrive/Motor_Vehicle_Collisions_-_Crashes_20250104.csv
📁 Model path: /content/drive/MyDrive/nyc_traffic_project/models/crash_predictor.pkl


In [18]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib
import os

# ---- 1. Load Data ----
print("📂 Loading data...")
data = pd.read_csv(DATA_PATH, low_memory=False)
data['CRASH DATE'] = pd.to_datetime(data['CRASH DATE'])
print(f"✅ Data loaded: {len(data):,} rows")

# ---- 2. Feature Engineering ----
print("🔧 Generating daily features...")
daily_crashes = data.groupby('CRASH DATE').size().reset_index(name='Crash_Count')
daily_crashes['Year'] = daily_crashes['CRASH DATE'].dt.year
daily_crashes['Month'] = daily_crashes['CRASH DATE'].dt.month
daily_crashes['Day'] = daily_crashes['CRASH DATE'].dt.day
daily_crashes['DayOfWeek'] = daily_crashes['CRASH DATE'].dt.dayofweek
print(f"📊 Generated {len(daily_crashes):,} daily records")

# ---- 3. Train Model ----
print("🧠 Training RandomForest model...")
X = daily_crashes[['Year', 'Month', 'Day', 'DayOfWeek']]
y = daily_crashes['Crash_Count']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("✅ Model training complete")

# ---- 4. Evaluate ----
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = model.score(X_test, y_test)
print(f"📊 Mean Absolute Error (MAE): {mae:.2f}")
print(f"📊 R² Score: {r2:.4f}")

# ---- 5. Save Model to Google Drive ----
joblib.dump(model, MODEL_PATH)
print(f"✅ Model saved to: {MODEL_PATH}")

# ---- 6. Download Model to Local (for GitHub upload) ----
from google.colab import files
files.download(MODEL_PATH)
print("📥 Model downloading to your local machine...")

📂 Loading data...
✅ Data loaded: 2,146,664 rows
🔧 Generating daily features...
📊 Generated 4,567 daily records
🧠 Training RandomForest model...
✅ Model training complete
📊 Mean Absolute Error (MAE): 40.08
📊 R² Score: 0.8944
✅ Model saved to: /content/drive/MyDrive/nyc_traffic_project/models/crash_predictor.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Model downloading to your local machine...


In [19]:
import joblib
import os

# 加载你刚保存的模型
model = joblib.load(MODEL_PATH)

# 验证它是否正常工作
print("✅ 模型加载成功！")
print(f"📊 模型类型: {type(model)}")
print(f"📊 特征重要性: {model.feature_importances_}")

# 做一个小预测
import pandas as pd
sample = pd.DataFrame({
    'Year': [2026],
    'Month': [9],
    'Day': [1],
    'DayOfWeek': [1]
})
pred = model.predict(sample)
print(f"🔮 预测 2026-09-01 事故数: {int(pred[0])}")

✅ 模型加载成功！
📊 模型类型: <class 'sklearn.ensemble._forest.RandomForestRegressor'>
📊 特征重要性: [0.81031351 0.05590616 0.06221863 0.0715617 ]
🔮 预测 2026-09-01 事故数: 247
